In [ ]:
# Standalone secret probe: shows the RAW HTTP answer of the Kaggle
# UserSecretsService endpoint, on every candidate host, with the status code
# and the server body preserved (kaggle_web_client hides both behind a
# generic ConnectionError).
#
# Findings (2026-10-19): api.kaggle.com -> HTTP 400, empty body.
#                        www.kaggle.com -> HTTP 200, real JSON body:
#                        {"errors":["No user secrets exist for kernel id N and
#                         label GEMINI_API_KEY."],"error":{"code":5},
#                         "wasSuccessful":false}   when the secret is not
#                        attached to this kernel.
import json, os, urllib.error, urllib.request

LABEL = 'GEMINI_API_KEY'
TOKEN = os.environ.get('KAGGLE_USER_SECRETS_TOKEN', '')
HOSTS = ['https://www.kaggle.com', 'https://api.kaggle.com']
ENDPOINT = '/requests/GetUserSecretByLabelRequest'
print('KAGGLE_URL_BASE injected by Kaggle:', os.environ.get('KAGGLE_URL_BASE'))
print('token present:', bool(TOKEN))
for host in HOSTS:
    body = json.dumps({'Label': LABEL}).encode()
    req = urllib.request.Request(
        host + ENDPOINT, data=body, method='POST',
        headers={'Content-type': 'application/json',
                 'X-Kaggle-Authorization': 'Bearer ' + TOKEN})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            status = resp.status
            payload = resp.read().decode('utf-8', 'replace')
    except urllib.error.HTTPError as exc:
        status = exc.code
        payload = exc.read().decode('utf-8', 'replace')
    except Exception as exc:
        status, payload = 0, f'{type(exc).__name__}: {exc}'
    try:
        shown = json.dumps(json.loads(payload), indent=1)[:800]
    except Exception:
        shown = payload[:800]
    print()
    print('==', host, '-> HTTP', status)
    print(shown)